# Lesson 16 — Convex Relaxations

## Learning objectives

1. Build the McCormick envelope of a bilinear term {cite:p}`McCormick1976`.
2. Recognize secant relaxations of univariate concave functions.
3. State the αBB convex underestimator {cite:p}`Adjiman1998,Adjiman1998a`.
4. Use `discopt`'s convex-relaxation infrastructure on a small NLP.

## 1. The McCormick envelope

For a bilinear term $z = x y$ with $x \in [\underline x, \overline x], y \in [\underline y, \overline y]$, the **convex hull** is the four McCormick inequalities:

$$
\begin{aligned}
z &\ge \underline x \, y + x \, \underline y - \underline x \, \underline y \\
z &\ge \overline x \, y + x \, \overline y - \overline x \, \overline y \\
z &\le \underline x \, y + x \, \overline y - \underline x \, \overline y \\
z &\le \overline x \, y + x \, \underline y - \overline x \, \underline y
\end{aligned}
$$

These give the *tightest* convex relaxation of the graph $\{(x, y, z) : z = xy\}$ over the box. Bound width determines tightness — narrower boxes → tighter envelopes.

## 2. Secant relaxation for concave $f$

For univariate concave $f$ on $[a, b]$:

- Tightest **convex underestimator**: the secant $f(a) + (f(b) - f(a))(x - a)/(b - a)$.
- Tightest **concave overestimator**: $f$ itself.

For convex $f$, swap the roles. Combined with McCormick for products, this handles **factorable** nonconvex problems {cite:p}`McCormick1976,Smith1999`. A function is *factorable* if it can be expressed as a finite composition of unary operators (exp, log, sin, $\cdot^k$) and binary operators ($+$, $\times$, $/$) applied to the variables; intuitively, anything you would write down by hand. Most engineering objectives and constraints are factorable; the convex-relaxation construction works **term-by-term** on the operator DAG of such a function. We revisit factorability formally in Lesson 22.

## 3. αBB underestimator for general C² functions

For any $f \in C^2$ on a box, the αBB underestimator is

$$\check f(x) = f(x) - \sum_i \alpha_i (\overline x_i - x_i)(x_i - \underline x_i)$$

with $\alpha_i \ge 0$ large enough to cancel any negative curvature of $f$. The subtracted term adds $2\alpha_i$ to the $i$-th diagonal of the Hessian, so a uniform $\alpha \ge \max\{0, -\tfrac12 \min_{x \in \text{box}} \lambda_{\min}(\nabla^2 f(x))\}$ suffices. Explicit choices use rigorous (interval) eigenvalue bounds on $\nabla^2 f$ over the box {cite:p}`Adjiman1998`. The maximum gap, $\sum_i \alpha_i (\overline x_i - \underline x_i)^2/4$ at the box centre, shrinks quadratically as the box is split. Convex by construction; tightness depends on $\alpha$.

This is the workhorse for global optimization of general factorable NLPs {cite:p}`Adjiman1998a,Tawarmalani2005`.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
# Bilinear: min x*y  s.t.  x + y >= 1, x,y in [0, 2].
# discopt automatically applies McCormick relaxations to bilinear terms
# inside its spatial-B&B; you don't construct them manually.
import numpy as np, discopt as do
from discopt.modeling import Model

m = Model("bilinear")
x = m.continuous("x", lb=0, ub=2)
y = m.continuous("y", lb=0, ub=2)
z = m.continuous("z", lb=-10, ub=10)              # auxiliary for the bilinear
m.subject_to(z == x * y)
m.subject_to(x + y >= 1)
m.minimize(z)

r = m.solve()
print(f"global z = {r.objective:.4f}; status: {r.status}; convex fast path: {r.convex_fast_path}")
print(f"dual bound = {r.bound}, certified = {r.gap_certified}, nodes = {r.node_count}")
# There is no stable public method that returns the root McCormick LP; it is
# assembled internally (discopt._relax.mccormick_lp / uniform_relax) and solved
# with the Rust simplex.


## 4. Tightness vs branching

The McCormick relaxation is exact at the box corners; loosest at the center. To tighten: **branch on $x$ (or $y$)** to split the box, recompute envelopes on each child. Spatial B&B (Lesson 21) is exactly this iterated.

The choice of which variable to branch on is the spatial branching rule; common heuristics are *most violating* and *largest box*.

## 5. Trilinear monomials

For products of *three* variables $z = x y w$ on a box, you can build a McCormick envelope by nesting: pick a pair ($x y$), introduce an auxiliary $u = x y$ with McCormick on $u$, then McCormick on $u w$. The catch is that the choice of which pair to nest first changes the envelope: there are three orderings and they don't agree.

Taking the tightest of all three nested orderings (pointwise max of the underestimators, min of the overestimators) is valid and at least as tight as any single ordering. It is still **not** the convex hull. The convex and concave envelopes of a trilinear monomial over a box are polyhedral, with facets through the box vertices that nested McCormick misses {cite:p}`rikun-1997,meyer-floudas-2004`. A full convex-hull characterization for general box domains has only recently been completed {cite:p}`makhoul-speakman-2025`.

`discopt._relax.envelopes` has all three: `relax_trilinear` (one fixed nesting order, the default path), `relax_trilinear_exact` (best of the three nestings; despite the name, *not* the hull, as its docstring says), and `relax_trilinear_meyer_floudas` (the vertex-facet hull envelope merged with the nested bounds). Inside a solve, the hull envelope is opt-in, with `m.solve(tuning=do.SolverTuning(trilinear_meyer=True))` (or the environment variable `DISCOPT_TRILINEAR=meyer`), and trilinear RLT rows are on by default.

## 6. αBB and eigenvalue bounds in `discopt`

The αBB construction needs $\alpha_i$ large enough to cancel the most negative curvature of $f$ over the box. `discopt` computes this **rigorously**: `discopt._alphabb_rigorous.rigorous_alpha` takes a sound *interval* enclosure of the Hessian over the box (`discopt._relax.convexity.interval_ad`) and applies an interval Gershgorin bound {cite:p}`Gershgorin1931` row by row:

$$\lambda_{\min} \ge H_{ii}^{\text{lo}} - \sum_{j \ne i} \max(|H_{ij}^{\text{lo}}|, |H_{ij}^{\text{hi}}|), \qquad \alpha_i = \max\{0, -\tfrac12\,(\text{that bound})\}.$$

Rigour matters. Estimating $\alpha$ from sampled Hessians (`discopt._relax.alphabb.estimate_alpha`) can miss the worst point in the box and give an *invalid* underestimator, so the sampled estimate is not used for bounds.

The same interval-eigenvalue machinery gives a **convexity certificate** (`discopt._relax.convexity.certificate`, `eigenvalue.py`). If the lower eigenvalue bound is $\ge 0$ on the box, $f$ is provably convex there. The convexity detector first runs a syntactic, disciplined-convex-programming walk over the expression DAG (`discopt._relax.convexity.classify_expr` / `classify_model`) and falls back to the certificate only when that walk says "unknown". A model proven convex skips spatial branching (`r.convex_fast_path == True`). `m.convexity()` asks the same question *before* you solve: it runs the same exact rewrites the solver applies first (for example $x \log x \to$ `entropy(x)`) and the same classifier, and reports a verdict per constraint (cell below). `is_convex=False` means *not proven*, not *proven nonconvex*: the classifier is sound but incomplete. The `convexity_detection` notebook in the `discopt` docs walks through the rules.

In [ ]:
# Ask for a convexity verdict before solving.
m = do.Model("convexity_demo")
x = m.continuous("x", lb=0.1, ub=4)
y = m.continuous("y", lb=0.1, ub=4)
m.subject_to(x**2 + y**2 <= 9, name="disk")
m.subject_to(x * y >= 1, name="hyperbola")
m.minimize(x * dm.log(x) + y)

rep = m.convexity()
print("convex:", rep.is_convex, "| rewrites:", rep.rewrites)
print("objective proven convex:", rep.objective_convex)
print("not proven convex:", rep.nonconvex_constraints())
r = m.solve()
print(f"route: {r.algorithm_route}  (convex fast path: {r.convex_fast_path})")
# x*y >= 1 with x, y > 0 *is* a convex set (the region above a hyperbola), but
# the classifier cannot prove it from the bilinear form, so the solve branches.
# Rewriting it as y >= 1/x makes the proof syntactic; try it.


## 7. RLT (Reformulation-Linearization Technique)

A more aggressive relaxation: multiply pairs of constraints together
(linear × linear → bilinear, then McCormick on each). Tightens the LP
relaxation at the cost of a quadratic blow-up in size {cite:p}`Sherali1990`.

## References
{cite:p}`McCormick1976,Adjiman1998,Adjiman1998a,Tawarmalani2005,Sherali1990,rikun-1997,meyer-floudas-2004,makhoul-speakman-2025`.